In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from collections import deque

class SecureTradingEnv(gym.Env):

    def __init__(
        self,
        data,
        assets,
        initial_balance=1_000_000,
        transaction_cost=0.001,
        slippage=0.0002,
        leverage=1.0, # <--- CHANGE THIS TO 1.0
        stop_loss=0.08,
        take_profit=0.20,
    ):
        super().__init__()

        self.data = data
        self.assets = assets
        self.initial_balance = initial_balance
        self.transaction_cost = transaction_cost
        self.slippage = slippage
        self.leverage = leverage
        self.stop_loss = stop_loss
        self.take_profit = take_profit
        self.n_assets = len(assets)
        self.prev_action = np.zeros(self.n_assets, dtype=np.float32)
        self.recent_returns = deque(maxlen=20) 
        self.recent_returns.append(0.0)

        self.action_space = spaces.Box(
            low=-1, high=1, shape=(self.n_assets,), dtype=np.float32
        )

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.n_assets * 13 + 2,),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.step_idx = 0
        self.cash = self.initial_balance
        self.positions = np.zeros(self.n_assets, dtype=np.float32)
        self.entry_prices = np.zeros(self.n_assets, dtype=np.float32)
        self.net_worth = self.initial_balance
        self.prev_net_worth = self.initial_balance
        self.peak_worth = self.initial_balance
        self.hold_steps = np.zeros(self.n_assets)

        return self._get_obs(), {}

    def _get_obs(self):
        obs = []

        for i, asset in enumerate(self.assets):
            row = self.data[asset].iloc[self.step_idx]
            
            # 1. Price Scaling: Scale price relative to initial price or a window 
            # This keeps the 'Close' signal within a digestible range for the NN
            scaled_close = row["Close"] / self.data[asset].iloc[0]["Close"]
            
            # 2. Bounded Indicators: Indicators like RSI are already 0-100
            norm_rsi = row["RSI"] / 100.0
            
            # 3. Trend/Volatility Indicators: Normalize by a rolling factor or ATR
            # We use a small epsilon (1e-8) to avoid division by zero
            norm_macd = row["MACD"] / (row["ATR"] + 1e-8)
            norm_atr = row["ATR"] / row["Close"]
            
            asset_features = [
                scaled_close,
                norm_rsi,
                norm_macd,
                row["MACD_signal"] / (row["ATR"] + 1e-8),
                row["SMA20"] / row["Close"], # Express SMA as a ratio to price
                row["EMA20"] / row["Close"],
                row["EMA50"] / row["Close"],
                (row["BB_upper"] - row["Close"]) / (row["ATR"] + 1e-8), # Dist from bands
                (row["BB_lower"] - row["Close"]) / (row["ATR"] + 1e-8),
                norm_atr,
                row["Momentum"] / row["Close"],
                np.tanh(row["OBV"] / 1e7), # Use tanh to squash large volume numbers
                row["Sentiment"] # Already -1 to 1 based on your plan
            ]
            obs.extend(asset_features)
    
        # 4. Portfolio Context: Always normalize these by initial balance
        obs.append(self.cash / self.initial_balance)
        obs.append(self.net_worth / self.initial_balance)
        return np.array(obs, dtype=np.float32)

    def step(self, action):
        # Scale Tanh output [-1, 1] to [0, 1] for long-only positions
        action = (np.array(action) + 1.0) / 2.0
        action = np.clip(action, 0, 1) # Failsafe
        total_trade_cost = 0.0
        prev_networth = self.net_worth
        reward = 0.0
        trade_happened = False
        # Calculate how much the action changed from the previous step
        action_diff = np.mean(np.abs(action - self.prev_action))
        # Penalize "jittery" behavior (e.g., a 0.01 penalty)
        reward -= 0.01 * action_diff 
        self.prev_action = action.copy() # Store this in __init__ as zeros
    
        # ===== FORCE CAPITAL USAGE =====
        min_exposure = 0.25
        current_exposure = np.sum(np.abs(self.positions))
    
        # ===== FORCE CAPITAL USAGE =====
        # Calculate max capital allowed per asset based on net worth and leverage
        # ===== FORCE CAPITAL USAGE =====
        max_capital_per_asset = (self.net_worth * self.leverage) / self.n_assets
    
        for i, asset in enumerate(self.assets):
            price = float(self.data[asset].iloc[self.step_idx]["Close"].item())
            price *= (1 + np.random.uniform(-self.slippage, self.slippage))
    
            # --- THE POSITION SIZING FIX ---
            target_capital = action[i] * max_capital_per_asset
            
            # Use np.floor for real-world whole shares
            target_position = np.floor(target_capital / price)
            position_change = target_position - self.positions[i]
            # -------------------------------
    
            # ===== TRADE EXECUTION (Fixed Buy/Sell Split) =====
            if abs(position_change) > (0.05 * max(self.positions[i], 1)):
                trade_happened = True
                trade_value = abs(position_change) * price
                fee = trade_value * self.transaction_cost
                total_trade_cost += fee

                if position_change > 0:  # BUYING
                    # REMOVED margin_required. Strict cash accounting.
                    total_cost = trade_value + fee 
                    
                    if total_cost <= self.cash:
                        self.cash -= total_cost
                        self.positions[i] = target_position
                        self.entry_prices[i] = price
                        self.hold_steps[i] = 0
                
                elif position_change < 0:  # SELLING
                    # Add proceeds back to cash (minus fee)
                    self.cash += (trade_value - fee)
                    self.positions[i] = target_position
                    
                    if self.positions[i] == 0:
                        self.entry_prices[i] = 0
                        self.hold_steps[i] = 0

            # ===== CLOSE TRADE =====
            if self.positions[i] != 0:
                pnl = (price - self.entry_prices[i]) * self.positions[i]
                pnl_pct = pnl / (self.entry_prices[i] * abs(self.positions[i]))
    
                if pnl_pct <= -self.stop_loss or pnl_pct >= self.take_profit:
                    # Current total value of the position
                    trade_value = self.positions[i] * price 
                    fee = trade_value * self.transaction_cost
                    
                    # Add proceeds to cash (Value minus exit fee)
                    self.cash += (trade_value - fee)
                    total_trade_cost += fee
                    
                    self.positions[i] = 0
                    self.entry_prices[i] = 0
                    self.hold_steps[i] = 0
    
        # ===== NET WORTH =====
        self.net_worth = self.cash + np.sum(
            self.positions * np.array([
                float(self.data[a].iloc[self.step_idx]["Close"].item())
                for a in self.assets
            ])
        )
        # 1. Calculate Core Metrics
        # 1. Calculate Core Metrics
        portfolio_return = np.log(self.net_worth / (prev_networth + 1e-9))
        self.recent_returns.append(portfolio_return)
        
        self.peak_worth = max(self.peak_worth, self.net_worth)
        drawdown = (self.peak_worth - self.net_worth) / (self.peak_worth + 1e-8)
        
        # ==========================================
        #       SHARPE RATIO OPTIMIZED REWARD
        # ==========================================
        # 1. Base Return
        reward = portfolio_return * 100 
        
        # 2. Local Volatility tracking
        if len(self.recent_returns) > 5:
            local_volatility = np.std(self.recent_returns)
            # Inject a micro-reward for maintaining a positive Local Sharpe
            local_sharpe = np.mean(self.recent_returns) / (local_volatility + 1e-9)
            reward += np.clip(local_sharpe * 0.05, -0.5, 0.5) 
        else:
            local_volatility = 0.0
            
        # 3. Penalties (Tuned for Secure, Low-Turnover Trading)
        # Increase cost penalty heavily to stop day-trading
        cost_penalty = (total_trade_cost / self.net_worth) * 50 
        
        # Massive penalty for flipping actions back and forth (Jitter)
        jitter_penalty = 0.5 * action_diff 
        
        # Directly punish high volatility to shrink the denominator of the Sharpe equation
        volatility_penalty = 5.0 * local_volatility 
        
        # REMOVED the holding_penalty entirely. We WANT the agent to hold!
        
        reward -= (drawdown * 2.0 + cost_penalty + jitter_penalty + volatility_penalty)
        # ==========================================
        
        # 3. Finalize Step
        self.prev_action = action.copy()
        self.step_idx += 1
        done = self.step_idx >= len(self.data[self.assets[0]]) - 1
    
        reward = float(np.clip(reward, -10, 10))
        return self._get_obs(), reward, done, False, {}

